# OriginalWoodelf — Depth Sweep (Notebook 2 / 3)

Runs **OriginalWoodelf** (`woodelf.simple_woodelf` — cube-based AAAI algorithm)
across all datasets and task types in the `woodelfhd_depth_sweep_experiment`.  
This notebook is **independent** and can run in parallel with Notebooks 01 and 03.
Notebook 04 merges all three result files and generates the HTML report.

### What this notebook does
1. Mounts Google Drive (results are saved there after each mission)
2. Clones `treebranchmarks` and `woodelf_explainer` repos
3. Installs all dependencies
4. Runs `woodelfhd_depth_sweep_experiment --method original_woodelf`
5. Writes partial results to Drive as `partial_original_woodelf.json`

### Datasets (all download automatically)
| Dataset | Source |
|---------|--------|
| Fraud Detection | Google Drive parquet (~200 MB) |
| HIGGS | UCI direct download (~8 GB uncompressed — takes ~20 min) |
| KDD Cup (Intrusion Detection) | Google Drive parquet |
| California Housing | sklearn builtin |

> **Runtime estimate:** OriginalWoodelf crashes at high depths (D≥18 for SHAP, D≥15
> for interactions), so most deep-depth entries are MEMORY_CRASH (instant).  
> Expect a few hours on a standard Colab CPU runtime.

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Step 2: Configure paths ──────────────────────────────────────────────────
import pathlib

DRIVE_FOLDER = pathlib.Path('/content/drive/MyDrive/treebranchmarks/woodelfhd_sweep')
DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)

DRIVE_RESULT_PATH = DRIVE_FOLDER / 'original_woodelf.json'
print(f'Method cache will be saved to: {DRIVE_RESULT_PATH}')

In [ ]:
# ── Step 3: Clone repositories ───────────────────────────────────────────────
TREEBRANCHMARKS_URL = 'YOUR_TREEBRANCHMARKS_REPO_URL'
WOODELF_URL         = 'YOUR_WOODELF_REPO_URL'

!git clone {TREEBRANCHMARKS_URL} /content/treebranchmarks
!git clone {WOODELF_URL}         /content/woodelf_explainer

In [ ]:
# ── Step 4: Install packages ─────────────────────────────────────────────────
!pip install -q -e /content/woodelf_explainer
!pip install -q -e /content/treebranchmarks

In [ ]:
# ── Step 5: Restore method cache from a previous interrupted run ─────────────
import shutil, pathlib

cache_dir = pathlib.Path('/content/treebranchmarks/cache/method_results/woodelfhd_depth_sweep_experiment')
cache_dir.mkdir(parents=True, exist_ok=True)
local_cache_file = cache_dir / 'original_woodelf.json'

if DRIVE_RESULT_PATH.exists() and not local_cache_file.exists():
    shutil.copy(DRIVE_RESULT_PATH, local_cache_file)
    print(f'Restored method cache ({DRIVE_RESULT_PATH.stat().st_size // 1024} KB)')
else:
    print('No method cache to restore — starting fresh.')

In [ ]:
# ── Step 6: Run the experiment (OriginalWoodelf only) ─────────────────────────
# --method original_woodelf : only the OriginalWoodelfApproach is timed
# The method name 'original_woodelf' matches ORIGINAL_WOODELF.name in builtin.py.

%cd /content/treebranchmarks

!python -u -m benchmarks.woodelfhd_depth_sweep_experiment \
    --method original_woodelf \
    --result_location "{DRIVE_RESULT_PATH}"

In [ ]:
# ── Step 7: Verify output ────────────────────────────────────────────────────
import json

with open(DRIVE_RESULT_PATH) as f:
    cache = json.load(f)

print(f'Entries in method cache: {len(cache)}')
if cache:
    sample = next(iter(cache.values()))
    print(f'Sample entry: {sample["_label"]}  →  {sample["running_time"]:.3f}s')
print(f'\nFile saved to: {DRIVE_RESULT_PATH}')